# FiLo natural-corruption robustness evaluation

Run this notebook with a Kaggle GPU and Internet enabled. It uses the official FiLo repository and the released cross-dataset FiLo + Grounding DINO checkpoints linked from its README. The selected pair is roughly 3.3 GB, before the OpenAI CLIP backbone cache. FiLo's official inference is single-image, so keep `BATCH_SIZE = 1`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("====== STEP 1: CLONING BENCHMARK AND OFFICIAL FiLo ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
FILO_ROOT = Path("/kaggle/working/FiLo")

def clone_or_update(url, destination):
    if not destination.exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)
    else:
        subprocess.run(["git", "-C", str(destination), "pull", "--ff-only"], check=True)

clone_or_update(
    f"https://github.com/{BENCHMARK_REPOSITORY}.git", BENCHMARK_ROOT
)
clone_or_update("https://github.com/CASIA-LMC-Lab/FiLo.git", FILO_ROOT)

print("\n====== STEP 2: INSTALLING FiLo DEPENDENCIES ======")
# Keep Kaggle's CUDA-enabled torch/torchvision. These are the FiLo runtime
# packages whose old APIs are used by the released source.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "addict==2.4.0",
        "ftfy==6.1.1", "huggingface-hub>=0.24,<1.0",
        "opencv-python-headless>=4.8", "pycocotools>=2.0.7",
        "regex>=2023.12", "scipy>=1.9", "Wand>=0.6",
        "scikit-image>=0.21", "scikit-learn>=1.3",
        "timm==0.9.16",
        "tokenizers==0.15.2", "transformers==4.38.1",
        "yapf==0.40.2", "einops>=0.7"
    ],
    check=True,
)
required_files = [
    FILO_ROOT / "models" / "FiLo.py",
    FILO_ROOT / "models" / "GroundingDINO" / "groundingdino"
    / "config" / "GroundingDINO_SwinT_OGC.py",
]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Incomplete FiLo clone; missing: {missing_files}")
os.environ["FILO_ROOT"] = str(FILO_ROOT)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"Benchmark repository: {BENCHMARK_ROOT}")
print(f"Official FiLo:        {FILO_ROOT}")
print("Environment ready.")

In [ ]:
import gc
import os
import sys
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download

BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
HARNESS_ROOT = BENCHMARK_ROOT / "zero_shot"
FILO_ROOT = Path("/kaggle/working/FiLo")
for import_path in (BENCHMARK_ROOT, HARNESS_ROOT, FILO_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["FILO_ROOT"] = str(FILO_ROOT)

from shared import corruption_plan_path
from harness.config import CLEAN_CONDITION
from harness.runner import run_evaluation

# Choose exactly one evaluation target.
# DATASET_NAME = "mvtec"
DATASET_NAME = "visa"
MODEL_NAME = "FiLo"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa"}:
    raise ValueError("DATASET_NAME must be either 'mvtec' or 'visa'.")
IS_MVTEC = DATASET_NAME == "mvtec"
SELECTED_DATASET = "MVTec AD" if IS_MVTEC else "VisA"
# Official zero-shot protocol: train on the other benchmark.
WEIGHT_DATASET = "visa" if IS_MVTEC else "mvtec"

USE_CATEGORIZED_CORRUPTIONS = True
CATEGORIZED_CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "motion_blur", "zoom_blur",
    "brightness", "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = ["noise", "blur", "photometric", "geometric"]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS
    else UNCATEGORIZED_CORRUPTION_TYPES
)
INCLUDE_CLEAN_BASELINE = True
ZERO_CORRUPTION_CONDITION = CLEAN_CONDITION
SEVERITY_LEVELS = [1, 2, 3, 4]
BATCH_SIZE = 1  # Required by the released FiLo forward path.
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"

MVTEC_PATH = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_PATH = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("FiLo's Grounding DINO stage requires a Kaggle GPU runtime.")

CORRUPTION_PLAN = corruption_plan_path(DATASET_NAME)
if USE_CATEGORIZED_CORRUPTIONS and not CORRUPTION_PLAN.exists():
    raise FileNotFoundError(f"Categorized corruption plan not found: {CORRUPTION_PLAN}")

HF_REPOSITORY = "FantasticGNU/FiLo"
CHECKPOINT_DIR = Path("/kaggle/working/filo_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_released_checkpoint(filename):
    # Reuse an attached Kaggle input first, which supports Internet-off reruns.
    input_root = Path("/kaggle/input")
    if input_root.exists():
        attached = next(input_root.rglob(filename), None)
        if attached is not None:
            return attached
    local_file = CHECKPOINT_DIR / filename
    if local_file.is_file():
        return local_file
    print(f"Downloading released checkpoint: {filename}")
    try:
        return Path(
            hf_hub_download(
                repo_id=HF_REPOSITORY,
                filename=filename,
                local_dir=str(CHECKPOINT_DIR),
                local_dir_use_symlinks=False,
            )
        )
    except Exception as exc:
        raise FileNotFoundError(
            f"Could not obtain {filename}. Enable Kaggle Internet or attach "
            f"that file from {HF_REPOSITORY}. Original error: {exc!r}"
        ) from exc

FILO_CHECKPOINT = resolve_released_checkpoint(
    f"filo_train_on_{WEIGHT_DATASET}.pth"
)
GROUNDING_CHECKPOINT = resolve_released_checkpoint(
    f"grounding_train_on_{WEIGHT_DATASET}.pth"
)
GROUNDING_CONFIG = (
    FILO_ROOT / "models" / "GroundingDINO" / "groundingdino"
    / "config" / "GroundingDINO_SwinT_OGC.py"
)

model_kwargs = {
    MODEL_NAME: {
        "filo_root": str(FILO_ROOT),
        "checkpoint_path": str(FILO_CHECKPOINT),
        "grounding_checkpoint_path": str(GROUNDING_CHECKPOINT),
        "groundingdino_config_path": str(GROUNDING_CONFIG),
        "dataset_name": DATASET_NAME,
        "clip_model": "ViT-L-14-336",
        "clip_pretrained": "openai",
        "image_size": 518,
        "features_list": [6, 12, 18, 24],
        "n_ctx": 12,
        "box_threshold": 0.25,
        "text_threshold": 0.25,
        "area_threshold": 0.7,
        "outside_box_weight": 0.7,
    }
}

print("LAUNCHING FiLo ROBUSTNESS BENCHMARK")
print(f"Evaluation target: {SELECTED_DATASET}")
print(f"Weights trained on: {WEIGHT_DATASET} (cross-dataset zero-shot)")
print(f"FiLo checkpoint:     {FILO_CHECKPOINT}")
print(f"Grounding checkpoint:{GROUNDING_CHECKPOINT}")
print(f"Zero corruption:     {ZERO_CORRUPTION_CONDITION if INCLUDE_CLEAN_BASELINE else 'disabled'}")
print(f"Corruptions:         {CORRUPTION_TYPES} @ {SEVERITY_LEVELS}")
print(f"Categorized:         {USE_CATEGORIZED_CORRUPTIONS}")
print(f"Plan:                {CORRUPTION_PLAN}")
print(f"Device/batch:        {DEVICE} / {BATCH_SIZE}")
print(f"Outputs:             {OUTPUT_ROOT}")

run_evaluation(
    mvtec_root=MVTEC_PATH if IS_MVTEC else None,
    visa_root=None if IS_MVTEC else VISA_PATH,
    output_root=OUTPUT_ROOT,
    models=[MODEL_NAME],
    model_kwargs=model_kwargs,
    device=DEVICE,
    dataset=DATASET_NAME,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    include_clean=INCLUDE_CLEAN_BASELINE,
    batch_size=BATCH_SIZE,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    categorized_corruption_plans={DATASET_NAME: str(CORRUPTION_PLAN)},
    corruption_seed=(
        CATEGORIZED_CORRUPTION_SEED if USE_CATEGORIZED_CORRUPTIONS else None
    ),
)

gc.collect()
torch.cuda.empty_cache()
print(f"FiLo evaluation complete. Outputs: {OUTPUT_ROOT}")